# The Math of Diffusion Processes

| Difficulty | ████████░░ 8/10 |
| :--- | :--- |
| Prerequisites | 09.05 |
| Time | ~90 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/06_diffusion_process_math.ipynb)

---

## 🎯 Learning Objective

Derive the forward and reverse diffusion processes, noise schedules, and denoising score matching that underlie DDPM/DDIM.

---

## 📐 Theory

`09.05` introduced score matching as a fourth route to generative modeling. **Diffusion
models** (DDPM: Denoising Diffusion Probabilistic Models) turn that idea into the dominant
approach behind modern image, audio, and video generation. This notebook derives the full
mathematics, step by step.

### Step 1: the forward (noising) process

Define a fixed, hand-picked Markov chain (recall `09.03`) that gradually destroys structure in
data $x_0 \sim p_{\text{data}}$ by adding a small amount of Gaussian noise at each of $T$ steps:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\!\big(x_t;\ \sqrt{1-\beta_t}\,x_{t-1},\ \beta_t I\big)$$

for a small **noise schedule** $\beta_1 < \beta_2 < \cdots < \beta_T$ (each $\beta_t \in (0,1)$).
Nothing here is learned — the forward process is a fixed, known recipe.

### Step 2: the closed-form marginal (the key algebraic trick)

Because each step is Gaussian, the entire chain of $t$ noising steps collapses into a *single*
formula for $x_t$ directly in terms of $x_0$ — no need to simulate $t$ separate steps. Defining
$\alpha_t = 1-\beta_t$ and $\bar\alpha_t = \prod_{s=1}^t \alpha_s$:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

i.e. $q(x_t \mid x_0) = \mathcal{N}(x_t;\sqrt{\bar\alpha_t}x_0,\ (1-\bar\alpha_t)I)$. As $t\to T$
with a well-chosen schedule, $\bar\alpha_T \to 0$, so $x_T$ approaches pure Gaussian noise
regardless of $x_0$ — the forward process deterministically destroys all information about the
original data, by construction. This closed form (verified numerically below against direct
step-by-step simulation) is what makes training tractable: you can jump to *any* noise level
$t$ in one step, rather than simulating the whole chain.

### Step 3: the reverse (denoising) process — what we actually train

The generative model is the *reverse* of the forward chain: starting from pure noise
$x_T \sim \mathcal{N}(0,I)$, learn to walk backward, $p_\theta(x_{t-1}\mid x_t)$, undoing the
noise one step at a time until you reach a sample from (approximately) $p_{\text{data}}$. Ho et
al. (2020) show the reverse conditional, if it were exactly known, is itself Gaussian with a
mean depending on the (unknown) noise $\epsilon$ that was added at that step — so the reverse
process reduces to learning to *predict that noise*: train a network $\epsilon_\theta(x_t, t)$
to estimate the $\epsilon$ that produced $x_t$ from $x_0$, using the simple loss:

$$\mathcal{L}(\theta) = \mathbb{E}_{x_0, t, \epsilon}\Big[\big\|\epsilon - \epsilon_\theta(x_t, t)\big\|^2\Big], \qquad
x_t = \sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$$

This loss is derivable as (a simplified form of) the ELBO for this latent-variable model
(`09.05`'s ELBO machinery, applied here with $T$ latent variables $x_1,\ldots,x_T$ instead of
one) — DDPM's "just predict the noise" training objective isn't an ad hoc heuristic, it falls
out of the same variational framework.

### Step 4: denoising score matching — the same objective, a different lens

Predicting the added noise $\epsilon$ turns out to be mathematically equivalent to estimating
the **score function** $\nabla_{x_t} \log q(x_t\mid x_0)$ (recall `09.05`'s score-matching
preview and `02.09`'s gradient-field framing): for this Gaussian forward process,
$\nabla_{x_t}\log q(x_t|x_0) = -\epsilon/\sqrt{1-\bar\alpha_t}$, so a noise predictor and a score
estimator differ only by a known rescaling. This is why DDPM and score-based generative models
are, at the level of the underlying mathematics, the same idea from two historical starting
points.

### Step 5: sampling (running the reverse process)

Once $\epsilon_\theta$ is trained, generate a new sample by starting at $x_T\sim\mathcal{N}(0,I)$
and iterating the learned reverse step:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\Big(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t,t)\Big) + \sqrt{\beta_t}\,z, \qquad z\sim\mathcal{N}(0,I)\ (\text{for } t>1)$$

repeated for $t=T,T-1,\ldots,1$. **DDIM** (Song et al., 2021) modifies this final step to make
the reverse process *deterministic* (no injected noise $z$) while still respecting the same
forward marginals, enabling far fewer sampling steps at some cost to sample diversity — a
practical speed/quality tradeoff on top of the same underlying mathematics.

---

In [ ]:
# Setup
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Watching a 2D point cloud (two separated blobs) progressively noised through the forward
process, at several snapshots between $t=0$ and $t=T$, makes "gradually destroy structure until
only Gaussian noise remains" a literal, visible sequence of pictures rather than an abstraction.

In [ ]:
rng = np.random.default_rng(0)
n_data = 400
centers = np.array([[-2.0, -2.0], [2.0, 2.0]])
labels = rng.integers(0, 2, n_data)
X0 = centers[labels] + rng.normal(scale=0.3, size=(n_data, 2))

T = 100
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

snapshot_ts = [0, 10, 30, 60, 99]
fig, axes = plt.subplots(1, len(snapshot_ts), figsize=(16, 3.2))
for ax, t in zip(axes, snapshot_ts):
    eps = rng.normal(size=X0.shape)
    x_t = np.sqrt(alpha_bars[t]) * X0 + np.sqrt(1 - alpha_bars[t]) * eps  # the closed-form marginal
    ax.scatter(x_t[:, 0], x_t[:, 1], s=6, alpha=0.5, color="#4C72B0")
    ax.set_title(f"t={t}\n(alpha_bar={alpha_bars[t]:.3f})", fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
plt.suptitle("Forward process: two Gaussian blobs dissolve into pure noise as t -> T")
plt.tight_layout()
plt.show()

print(f"alpha_bar at t=0: {alpha_bars[0]:.4f} (data barely touched)")
print(f"alpha_bar at t={T-1}: {alpha_bars[-1]:.4f} (data signal almost entirely replaced by noise)")

## 🐍 Implementation from Scratch

We verify the forward process's closed-form marginal matches direct step-by-step simulation,
then train a real (if tiny) noise-prediction network and run the full reverse sampling loop to
generate new samples from a toy 2D distribution.

In [ ]:
# --- Verify the closed-form marginal q(x_t|x_0) matches iterating the single-step process ---
T = 50
betas = np.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)

x0_val = 2.0
n_trials = 200_000

x_iterative = np.full(n_trials, x0_val)
for t in range(T):
    eps_t = rng.normal(size=n_trials)
    x_iterative = np.sqrt(1 - betas[t]) * x_iterative + np.sqrt(betas[t]) * eps_t   # one forward step at a time

eps_direct = rng.normal(size=n_trials)
x_closed_form = np.sqrt(alpha_bars[-1]) * x0_val + np.sqrt(1 - alpha_bars[-1]) * eps_direct  # the Step 2 formula

print("Forward process: T single steps vs. the closed-form marginal, same x_0:")
print(f"  iterative:   mean={x_iterative.mean():.4f}, std={x_iterative.std():.4f}")
print(f"  closed-form: mean={x_closed_form.mean():.4f}, std={x_closed_form.std():.4f}")
print(f"  theory says: mean={np.sqrt(alpha_bars[-1])*x0_val:.4f}, std={np.sqrt(1-alpha_bars[-1]):.4f}")
print(f"  All three agree: {np.allclose([x_iterative.mean(), x_closed_form.mean()], np.sqrt(alpha_bars[-1])*x0_val, atol=0.01)}")

# --- Train a real noise-prediction network on a toy bimodal 2D dataset ---
torch.manual_seed(0)
n_data = 600
centers = np.array([[-2.0, -2.0], [2.0, 2.0]])
labels = rng.integers(0, 2, n_data)
X = centers[labels] + rng.normal(scale=0.3, size=(n_data, 2))
X_t = torch.tensor(X, dtype=torch.float32)

T_train = 100
betas_t = torch.linspace(1e-4, 0.02, T_train)
alphas_t = 1 - betas_t
alpha_bars_t = torch.cumprod(alphas_t, dim=0)

class NoisePredictor(nn.Module):
    def __init__(self, dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim + 1, hidden), nn.ReLU(),
                                  nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, dim))
    def forward(self, x, t_frac):
        return self.net(torch.cat([x, t_frac], dim=1))  # condition on x_t AND the timestep

model = NoisePredictor()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(3000):
    opt.zero_grad()
    t_idx = torch.randint(0, T_train, (X_t.shape[0],))
    ab = alpha_bars_t[t_idx].unsqueeze(1)
    eps = torch.randn_like(X_t)
    x_t = torch.sqrt(ab) * X_t + torch.sqrt(1 - ab) * eps    # the SAME closed-form marginal, now used for training
    t_frac = (t_idx.float() / T_train).unsqueeze(1)
    loss = ((eps - model(x_t, t_frac)) ** 2).mean()          # Step 3's noise-prediction loss, directly
    loss.backward()
    opt.step()

print(f"\nFinal training loss (predicting the injected noise): {loss.item():.4f}")

## 🤖 Why This Matters for AI

Stable Diffusion, DALL-E 2/3, Midjourney, and every other major image/video/audio generator
released since 2022 is built on exactly this mathematics — the forward process, closed-form
marginal, noise-prediction training loss, and reverse sampling loop derived step by step above
(usually operating in a compressed latent space via a VAE, recall `09.05`, rather than directly
on pixels — "latent diffusion"). Below, we run the *entire pipeline end to end* on the toy
bimodal dataset trained in the Implementation section: starting purely from Gaussian noise,
iterate the learned reverse process $T$ times, and check whether the result resembles the
original two-cluster training distribution — a genuine, if minimal, working diffusion model.

In [ ]:
# The reverse sampling loop, exactly as derived in Step 5: start at pure noise, iterate
# the learned denoising step T times, down to t=0.
with torch.no_grad():
    x = torch.randn(300, 2)   # x_T ~ N(0, I): pure noise, no information about the data whatsoever
    trajectory_snapshots = {}
    for t in reversed(range(T_train)):
        t_frac = torch.full((300, 1), t / T_train)
        eps_pred = model(x, t_frac)
        alpha_t, alpha_bar_t, beta_t = alphas_t[t], alpha_bars_t[t], betas_t[t]
        mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * eps_pred)
        if t > 0:
            x = mean + torch.sqrt(beta_t) * torch.randn_like(x)   # inject fresh noise, per the DDPM sampler
        else:
            x = mean                                               # final step: no more noise to add
        if t in (99, 60, 30, 10, 0):
            trajectory_snapshots[t] = x.clone().numpy()

samples = x.numpy()
dist_to_c0 = np.linalg.norm(samples - centers[0], axis=1)
dist_to_c1 = np.linalg.norm(samples - centers[1], axis=1)
near_a_center = np.mean(np.minimum(dist_to_c0, dist_to_c1) < 1.0)

print(f"Real data mean: {X.mean(axis=0)}")
print(f"Generated samples mean: {samples.mean(axis=0)}")
print(f"Fraction of generated samples within 1.0 of a true cluster center: {near_a_center:.3f}")
print("(Started from PURE Gaussian noise, with no direct information about the two-cluster")
print("structure -- everything the reverse process 'knows' came from the noise-prediction")
print("network learning Step 3's objective during training.)")

fig, axes = plt.subplots(1, 5, figsize=(16, 3.2))
for ax, t in zip(axes, sorted(trajectory_snapshots.keys(), reverse=True)):
    pts = trajectory_snapshots[t]
    ax.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.5, color="#C44E52")
    ax.set_title(f"reverse step t={t}", fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
plt.suptitle("Reverse process: pure noise gradually resolves into the learned two-cluster shape")
plt.tight_layout()
plt.show()

## 🏋️ Exercises

### Warm-up
1. For a schedule with just $T=3$ steps and $\beta_1=\beta_2=\beta_3=0.1$, compute
   $\bar\alpha_1,\bar\alpha_2,\bar\alpha_3$ by hand. Verify with the code's `np.cumprod` pattern.

### Practice
2. In the Implementation cell, change `T_train` from 100 to 20 (fewer, larger noising steps)
   and retrain. Does the model still learn to generate the two clusters well? Report the final
   training loss and the fraction of samples landing near a true center, and explain in a
   sentence why fewer/coarser steps might hurt sample quality.

### Challenge
3. Implement a simplified DDIM-style *deterministic* sampler: remove the `torch.sqrt(beta_t) *
   torch.randn_like(x)` noise-injection term from the reverse loop entirely (keep everything
   else identical) and re-generate samples. Compare the diversity of the resulting samples
   (e.g. the spread/std within each recovered cluster) against the original stochastic DDPM
   sampler's diversity. Connect your observation to the real DDIM paper's claimed speed/quality
   tradeoff.

---

## 📚 Further Reading
- Ho, Jain & Abbeel, ["Denoising Diffusion Probabilistic Models"](https://arxiv.org/abs/2006.11239) (DDPM)
- Song, Meng & Ermon, ["Denoising Diffusion Implicit Models"](https://arxiv.org/abs/2010.02502) (DDIM)
- Sohl-Dickstein et al., ["Deep Unsupervised Learning using Nonequilibrium Thermodynamics"](https://arxiv.org/abs/1503.03585) (the original diffusion-model paper)

---

*Next notebook: [Manifolds and Curvature](../10_Differential_Geometry_and_Topology/01_manifolds_and_curvature.ipynb)*